# 5-Step DuPont Decomposition — Greek Systemic Banks (2022–2024)

**Methodology:** Banking-adapted 5-step DuPont (Petersen & Plenborg, *Financial Statement Analysis*, 2012).
Standard corporate DuPont uses EBIT, which is undefined for banks. This notebook uses the banking equivalent:

$$\text{ROE} = \underbrace{\frac{\text{NP}}{\text{PBT}}}_{\text{Tax Burden}} \times \underbrace{\frac{\text{PBT}}{\text{PPOP}}}_{\text{Provision Burden}} \times \underbrace{\frac{\text{PPOP}}{\text{Op. Income}}}_{\text{Pre-provision Margin}} \times \underbrace{\frac{\text{Op. Income}}{\text{Assets}}}_{\text{Asset Utilisation}} \times \underbrace{\frac{\text{Assets}}{\text{Equity}}}_{\text{Leverage}}$$

where **PPOP** = Pre-Provision Operating Profit = PBT + |Impairment losses|.

**Sources:** Official annual reports (PDFs in `02_Banking_Sector_Dashboard/data/raw/`).  
**Data verified:** All figures cross-checked against source PDF pages (see `DATA_DICTIONARY.md`).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# ── Brand colors (bank logo palette) ──────────────────────────────────────────
COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS  = list(COLORS.keys())
YEARS  = [2022, 2023, 2024]

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

print('Libraries loaded. Data directory:', DATA_DIR.resolve())

Libraries loaded. Data directory: C:\Users\Spyro\Desktop\FinTech -Analyst\Projects\Greek_Banking_Sector_Analysis\02_Banking_Sector_Dashboard\data\processed


In [2]:
# ── Load processed data ───────────────────────────────────────────────────────
kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
is_long = pd.read_csv(DATA_DIR / 'income_statement_final.csv')

# Pivot income statement to extract Profit Before Tax (not in kpis_final)
is_pivot = (
    is_long
    .pivot_table(index=['bank', 'year'], columns='metric', values='value', aggfunc='first')
    .reset_index()
)
pbt_col = 'Profit before tax'
assert pbt_col in is_pivot.columns, f"Column '{pbt_col}' not found in income statement."

# Merge PBT into kpis
df = kpis.merge(
    is_pivot[['bank', 'year', pbt_col]].rename(columns={pbt_col: 'profit_before_tax'}),
    on=['bank', 'year'],
    how='left'
)

print(f'Loaded {len(df)} bank-year observations. Columns: {list(df.columns)}')
df[['bank', 'year', 'net_profit', 'profit_before_tax', 'impairment', 'operating_income', 'total_assets', 'equity', 'roe']].round(1)

Loaded 12 bank-year observations. Columns: ['bank', 'year', 'nii', 'operating_income', 'operating_expenses', 'net_profit', 'impairment', 'loans', 'deposits', 'equity', 'total_assets', 'roe', 'cost_to_income', 'loan_to_deposit', 'nim', 'nii_yoy', 'profit_yoy', 'assets_yoy', 'opincome_yoy', 'roa', 'cet1', 'npe_ratio', 'profit_before_tax']


,bank,year,net_profit,profit_before_tax,impairment,operating_income,total_assets,equity,roe
0,Alpha Bank,2022,342.2,563.2,-559.0,2194.0,77200.6,6199.6,5.5
1,Alpha Bank,2023,655.0,828.3,-381.0,2168.2,72694.1,7288.7,9.0
2,Alpha Bank,2024,668.0,871.0,-360.0,2272.0,70954.0,8155.0,8.2
3,Eurobank,2022,1336.0,1741.0,-292.0,3135.0,81471.0,6667.0,20.0
4,Eurobank,2023,1148.0,1550.0,-413.0,2914.0,79815.0,7532.0,15.2
5,Eurobank,2024,1458.0,1878.0,-305.0,3339.0,101151.0,8636.0,16.9
6,NBG,2022,1120.0,1155.0,-280.0,2355.0,78113.0,6475.0,17.3
7,NBG,2023,1106.0,1479.0,-268.0,2760.0,74584.0,7652.0,14.5
8,NBG,2024,1158.0,1517.0,-180.0,2885.0,74957.0,8452.0,13.7
9,Piraeus Bank,2022,864.0,943.0,-523.0,2516.0,75403.0,6512.0,13.3


In [3]:
# ── Compute 5-step DuPont components ─────────────────────────────────────────
#
# PPOP = Profit Before Tax + |Impairment losses|
# (impairment column in kpis is negative, so we subtract to add back)
#
df['ppop'] = df['profit_before_tax'] + df['impairment'].abs()

df['tax_burden']         = df['net_profit']       / df['profit_before_tax']  # (1) NP / PBT
df['provision_burden']   = df['profit_before_tax'] / df['ppop']              # (2) PBT / PPOP
df['preprovision_margin']= df['ppop']              / df['operating_income']   # (3) PPOP / OpInc
df['asset_utilisation']  = df['operating_income']  / df['total_assets']      # (4) OpInc / Assets
df['leverage']           = df['total_assets']      / df['equity']             # (5) Assets / Equity

# Reconstruct ROE and express as %
df['roe_reconstructed'] = (
    df['tax_burden']
    * df['provision_burden']
    * df['preprovision_margin']
    * df['asset_utilisation']
    * df['leverage']
    * 100
)

components = ['tax_burden', 'provision_burden', 'preprovision_margin', 'asset_utilisation', 'leverage']
print('DuPont components computed.')
df[['bank', 'year'] + components + ['roe_reconstructed', 'roe']].round(4)

DuPont components computed.


,bank,year,tax_burden,provision_burden,preprovision_margin,asset_utilisation,leverage,roe_reconstructed,roe
0,Alpha Bank,2022,0.6076,0.5019,0.5115,0.0284,12.4525,5.5197,5.5
1,Alpha Bank,2023,0.7908,0.6849,0.5577,0.0298,9.9735,8.9865,9.0
2,Alpha Bank,2024,0.7669,0.7076,0.5418,0.0320,8.7007,8.1913,8.2
3,Eurobank,2022,0.7674,0.8564,0.6485,0.0385,12.2200,20.0390,20.0
4,Eurobank,2023,0.7406,0.7896,0.6736,0.0365,10.5968,15.2416,15.2
5,Eurobank,2024,0.7764,0.8603,0.6538,0.0330,11.7127,16.8828,16.9
6,NBG,2022,0.9697,0.8049,0.6093,0.0301,12.0638,17.2973,17.3
7,NBG,2023,0.7478,0.8466,0.6330,0.0370,9.7470,14.4537,14.5
8,NBG,2024,0.7633,0.8939,0.5882,0.0385,8.8686,13.7009,13.7
9,Piraeus Bank,2022,0.9162,0.6432,0.5827,0.0334,11.5791,13.2678,13.3


In [4]:
# ── Sanity check: reconstructed ROE must match reported within ±0.5 pp ────────
df['roe_error_pp'] = (df['roe_reconstructed'] - df['roe']).abs()

MAX_TOLERANCE_PP = 0.5
failures = df[df['roe_error_pp'] > MAX_TOLERANCE_PP][['bank', 'year', 'roe', 'roe_reconstructed', 'roe_error_pp']]

print('\n── ROE Reconstruction Check (' + f'tolerance ±{MAX_TOLERANCE_PP}pp) ──')
print(df[['bank', 'year', 'roe', 'roe_reconstructed', 'roe_error_pp']].to_string(index=False))

if len(failures) > 0:
    print('\n⚠️  FAILURES:')
    print(failures.to_string(index=False))
    raise AssertionError(f'{len(failures)} bank-year(s) exceed ±{MAX_TOLERANCE_PP}pp tolerance. Investigate data.')
else:
    print(f'\n✅ All 12 bank-years reconstruct within ±{MAX_TOLERANCE_PP}pp.')


── ROE Reconstruction Check (tolerance ±0.5pp) ──
        bank  year  roe  roe_reconstructed  roe_error_pp
  Alpha Bank  2022  5.5           5.519711      0.019711
  Alpha Bank  2023  9.0           8.986513      0.013487
  Alpha Bank  2024  8.2           8.191294      0.008706
    Eurobank  2022 20.0          20.038998      0.038998
    Eurobank  2023 15.2          15.241636      0.041636
    Eurobank  2024 16.9          16.882816      0.017184
         NBG  2022 17.3          17.297297      0.002703
         NBG  2023 14.5          14.453738      0.046262
         NBG  2024 13.7          13.700899      0.000899
Piraeus Bank  2022 13.3          13.267813      0.032187
Piraeus Bank  2023 10.7          10.716714      0.016714
Piraeus Bank  2024 12.9          12.885289      0.014711

✅ All 12 bank-years reconstruct within ±0.5pp.


In [5]:
# ── Chart 1: Small Multiples — Component Evolution Per Bank ──────────────────
# For each bank: how did each DuPont lever change from 2022 → 2024?
# Normalised to 2022 = 100 so all 5 levers are comparable on one axis.

COMP_LABELS = {
    'tax_burden':          'Tax Burden (NP/PBT)',
    'provision_burden':    'Provision Burden (PBT/PPOP)',
    'preprovision_margin': 'Pre-provision Margin (PPOP/OpInc)',
    'asset_utilisation':   'Asset Utilisation (OpInc/Assets)',
    'leverage':            'Leverage (Assets/Equity)',
}
COMP_COLORS = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']

fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    shared_xaxes=False,
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    bank_df = df[df['bank'] == bank].sort_values('year')
    base = bank_df.set_index('year')[components].loc[2022]  # 2022 base

    for ci, (comp, color) in enumerate(zip(components, COMP_COLORS)):
        indexed = (bank_df[comp].values / base[comp]) * 100
        fig1.add_trace(
            go.Scatter(
                x=YEARS, y=indexed,
                name=COMP_LABELS[comp],
                line=dict(color=color, width=2),
                marker=dict(size=7),
                showlegend=(bi == 0),
                legendgroup=comp,
            ),
            row=row + 1, col=col + 1,
        )

    # Overlay reconstructed ROE trend (dashed)
    roe_indexed = (bank_df['roe_reconstructed'].values / bank_df['roe_reconstructed'].values[0]) * 100
    fig1.add_trace(
        go.Scatter(
            x=YEARS, y=roe_indexed,
            name='ROE (indexed)',
            line=dict(color='white', width=2.5, dash='dash'),
            marker=dict(size=8, symbol='diamond'),
            showlegend=(bi == 0),
            legendgroup='roe',
        ),
        row=row + 1, col=col + 1,
    )

    # Add 100 baseline
    fig1.add_hline(y=100, line_dash='dot', line_color='rgba(255,255,255,0.3)', row=row + 1, col=col + 1)

fig1.update_layout(
    title_text='<b>DuPont Component Evolution</b> | Indexed to 2022 = 100 | Greek Systemic Banks',
    title_font_size=16,
    template='plotly_dark',
    height=620,
    legend=dict(orientation='h', yanchor='bottom', y=-0.22, x=0.5, xanchor='center', font_size=11),
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig1.update_xaxes(tickvals=YEARS, ticktext=[str(y) for y in YEARS])
fig1.update_yaxes(title_text='Index (2022=100)', ticksuffix='')
fig1.show()

In [6]:
# ── Chart 2: Cross-Bank Comparison Per DuPont Lever (2024) ───────────────────
# For 2024: which bank leads on each lever? This is the equity-research comparison.

df24 = df[df['year'] == 2024].set_index('bank')

# Use 2×3 layout (5 components + ROE)
fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Tax Burden (NP/PBT)',
        'Provision Burden (PBT/PPOP)',
        'Pre-provision Margin (PPOP/OpInc)',
        'Asset Utilisation (OpInc/Assets, %)',
        'Leverage (Assets/Equity, ×)',
        'ROE (%, reported)',
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.10,
)

metrics = components + ['roe']
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]
multipliers = [1, 1, 1, 100, 1, 1]  # asset utilisation shown as %

for (metric, (r, c), mult) in zip(metrics, positions, multipliers):
    values = [df24.loc[b, metric] * mult for b in BANKS if b in df24.index]
    bar_colors = [COLORS[b] for b in BANKS if b in df24.index]
    fig2.add_trace(
        go.Bar(
            x=BANKS,
            y=values,
            marker_color=bar_colors,
            text=[f'{v:.2f}' if mult == 1 else f'{v:.2f}%' for v in values],
            textposition='outside',
            showlegend=False,
        ),
        row=r, col=c,
    )

fig2.update_layout(
    title_text='<b>DuPont Lever Comparison — 2024</b> | Greek Systemic Banks',
    title_font_size=16,
    template='plotly_dark',
    height=600,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig2.update_xaxes(tickfont_size=11)
fig2.show()

In [7]:
# ── Chart 3: ROE Bridge — What drove ROE differences in 2024? ────────────────
# Compare each bank vs sector median on each DuPont lever.
# Shows which bank is above / below peer median and by how much (in log-contribution).

df24_comp = df[df['year'] == 2024].copy()
sector_medians = df24_comp[components].median()

# Log-contribution relative to sector median (sum = ROE premium over median)
for comp in components:
    df24_comp[f'log_{comp}'] = np.log(df24_comp[comp] / sector_medians[comp])

log_cols = [f'log_{c}' for c in components]
df24_comp['total_log_premium'] = df24_comp[log_cols].sum(axis=1)

fig3 = go.Figure()
bar_width = 0.15
for bi, bank in enumerate(BANKS):
    row_b = df24_comp[df24_comp['bank'] == bank].iloc[0]
    log_vals = [row_b[f'log_{c}'] for c in components]
    x_pos = np.arange(len(components)) + bi * bar_width - (len(BANKS) - 1) * bar_width / 2
    fig3.add_trace(go.Bar(
        x=x_pos,
        y=log_vals,
        name=bank,
        marker_color=COLORS[bank],
        width=bar_width,
        text=[f'{v:+.3f}' for v in log_vals],
        textposition='outside',
        textfont_size=9,
    ))

fig3.add_hline(y=0, line_color='rgba(255,255,255,0.4)', line_dash='dash')
comp_abbrev = ['Tax Burden', 'Provision Burden', 'Pre-prov. Margin', 'Asset Util.', 'Leverage']
fig3.update_layout(
    title_text='<b>ROE Driver Analysis — 2024</b> | Log premium/discount vs sector median per DuPont lever',
    title_font_size=15,
    template='plotly_dark',
    height=430,
    xaxis=dict(
        tickmode='array',
        tickvals=np.arange(len(components)),
        ticktext=comp_abbrev,
    ),
    yaxis_title='Log premium vs sector median (+ = above median)',
    legend=dict(orientation='h', y=-0.20, x=0.5, xanchor='center'),
    barmode='group',
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig3.show()

In [8]:
# ── Key Insights Table ────────────────────────────────────────────────────────
print('=' * 72)
print('  DuPont Summary — Greek Systemic Banks (2022–2024)')
print('=' * 72)

display_cols = [
    ('bank', 'Bank'), ('year', 'Year'),
    ('tax_burden', 'Tax Burden'), ('provision_burden', 'Prov. Burden'),
    ('preprovision_margin', 'PreProv. Margin'), ('asset_utilisation', 'Asset Util. (%)'),
    ('leverage', 'Leverage (×)'), ('roe', 'ROE (%)'),
]

out = df[['bank','year','tax_burden','provision_burden','preprovision_margin','asset_utilisation','leverage','roe']].copy()
out['asset_utilisation'] = (out['asset_utilisation'] * 100).round(2)
out[['tax_burden','provision_burden','preprovision_margin']] = out[['tax_burden','provision_burden','preprovision_margin']].round(3)
out['leverage'] = out['leverage'].round(2)
out['roe'] = out['roe'].round(1)

print(out.to_string(index=False))

print('\n── Key Findings ──')

# Find who has the highest/lowest of each component in 2024
for comp, label in zip(components, comp_abbrev):
    mult = 100 if comp == 'asset_utilisation' else 1
    best_bank = df24_comp.set_index('bank')[comp].idxmax()
    best_val  = df24_comp.set_index('bank')[comp].max() * mult
    worst_bank = df24_comp.set_index('bank')[comp].idxmin()
    worst_val  = df24_comp.set_index('bank')[comp].min() * mult
    unit = '%' if comp == 'asset_utilisation' else '×' if comp == 'leverage' else ''
    fmt = '.2f' if comp in ('asset_utilisation', 'leverage') else '.3f'
    print(f'  {label:<22}  Best: {best_bank} ({best_val:{fmt}}{unit})   '
          f'Worst: {worst_bank} ({worst_val:{fmt}}{unit})')

  DuPont Summary — Greek Systemic Banks (2022–2024)
        bank  year  tax_burden  provision_burden  preprovision_margin  asset_utilisation  leverage  roe
  Alpha Bank  2022       0.608             0.502                0.511               2.84     12.45  5.5
  Alpha Bank  2023       0.791             0.685                0.558               2.98      9.97  9.0
  Alpha Bank  2024       0.767             0.708                0.542               3.20      8.70  8.2
    Eurobank  2022       0.767             0.856                0.648               3.85     12.22 20.0
    Eurobank  2023       0.741             0.790                0.674               3.65     10.60 15.2
    Eurobank  2024       0.776             0.860                0.654               3.30     11.71 16.9
         NBG  2022       0.970             0.805                0.609               3.01     12.06 17.3
         NBG  2023       0.748             0.847                0.633               3.70      9.75 14.5
         NBG

In [9]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12,              'Expected 12 bank-year rows (4 banks × 3 years).'
assert df['profit_before_tax'].notna().all(), 'Missing profit_before_tax values.'
assert df['roe_error_pp'].max() < 0.5,        'ROE reconstruction error exceeds ±0.5pp.'

# Component sanity: all multipliers should be positive
for comp in components:
    assert (df[comp] > 0).all(), f'Non-positive value in component: {comp}'

print('✅ All checks passed.')
print(f'   Max ROE reconstruction error: {df["roe_error_pp"].max():.4f} pp')
print(f'   Mean ROE reconstruction error: {df["roe_error_pp"].mean():.4f} pp')

✅ All checks passed.
   Max ROE reconstruction error: 0.0463 pp
   Mean ROE reconstruction error: 0.0211 pp
